# MA3632 — Workshop 11: Pipelines and Reproducible Workflows

This workshop accompanies Lecture 11. We demonstrate the data leakage problem
numerically, build pipelines that prevent it, handle heterogeneous feature types
with column transformers, run hyperparameter search over a full pipeline, and
save and reload a fitted pipeline with `joblib`. Part E implements a custom
transformer from scratch.

Work through all parts in order. Take-home exercises are at the end.

---

## Part A — Data leakage in practice

We measure the optimism introduced by fitting a scaler on the full dataset
before cross-validation, compared with the correct approach of fitting inside
each fold via a pipeline.

### A1. Imports and data

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import joblib, warnings
warnings.filterwarnings("ignore")

from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    cross_val_score, StratifiedKFold, GridSearchCV, train_test_split
)
from sklearn.metrics import accuracy_score
from sklearn.base import BaseEstimator, TransformerMixin

digits = load_digits()
X_dig, y_dig = digits.data, digits.target

X_tr, X_te, y_tr, y_te = train_test_split(
    X_dig, y_dig, test_size=0.25, random_state=0, stratify=y_dig
)
print(f"Digits — train: {X_tr.shape}, test: {X_te.shape}")

### A2. Leaky vs correct cross-validation

In [ ]:
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=1)

# --- Leaky approach ---
# Scaler fitted on ALL training data before CV; validation fold has already
# contributed to the scaler parameters.
scaler_leaky = StandardScaler()
X_tr_scaled_leaky = scaler_leaky.fit_transform(X_tr)
leaky_scores = cross_val_score(
    LogisticRegression(max_iter=500), X_tr_scaled_leaky, y_tr,
    cv=cv, scoring='accuracy'
)

# --- Correct approach ---
# Scaler is a pipeline step; it is re-fitted fresh on each training fold.
pipe_correct = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(max_iter=500)),
])
correct_scores = cross_val_score(pipe_correct, X_tr, y_tr, cv=cv, scoring='accuracy')

print(f"Leaky CV     — mean: {leaky_scores.mean():.4f}  std: {leaky_scores.std():.4f}")
print(f"Correct CV   — mean: {correct_scores.mean():.4f}  std: {correct_scores.std():.4f}")
print(f"Optimism bias: {leaky_scores.mean() - correct_scores.mean():+.4f}")

In [ ]:
# Visualise fold-by-fold difference
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(range(1, 11), leaky_scores,   'o-', color='tomato',    label='Leaky (scaler before CV)')
ax.plot(range(1, 11), correct_scores, 's-', color='steelblue', label='Correct (pipeline CV)')
ax.set_xlabel('Fold')
ax.set_ylabel('Accuracy')
ax.set_title('Leaky vs correct CV — Logistic Regression on Digits')
ax.legend()
plt.tight_layout()
plt.savefig('/tmp/a2_leakage.png', dpi=110)
plt.close()
print("Saved a2_leakage.png")

**Exercise A.** The optimism bias is small on the Digits dataset because the
class distributions are well balanced and the dataset is large. Describe a
situation in which the bias would be larger, and explain which property of the
dataset drives the size of the bias.

### A3. A second kind of leakage: feature selection

Scaling is not the only step that can leak. If features are *selected* using
the full dataset before cross-validation, the validation fold's labels have
already influenced which features the model is allowed to see. This is
usually far more damaging than scaler leakage, especially when the number of
features is large relative to the number of observations.

In [ ]:
# Pure noise: n=80 observations, p=2000 features, none of which relate to y.
# True generalisation accuracy of any classifier here should be at chance (0.50).
rng_fs = np.random.default_rng(20)
n_fs, p_fs = 80, 2000
X_noise = rng_fs.standard_normal((n_fs, p_fs))
y_noise = rng_fs.integers(0, 2, size=n_fs)

cv_fs = StratifiedKFold(n_splits=5, shuffle=True, random_state=21)

# --- Leaky approach: select top-15 features by correlation with y on ALL data first ---
selector_leaky = SelectKBest(f_classif, k=15)
X_selected_leaky = selector_leaky.fit_transform(X_noise, y_noise)
leaky_fs_scores = cross_val_score(
    LogisticRegression(max_iter=500), X_selected_leaky, y_noise,
    cv=cv_fs, scoring='accuracy'
)

# --- Correct approach: selection is a pipeline step, refit inside each fold ---
pipe_fs_correct = Pipeline([
    ('select', SelectKBest(f_classif, k=15)),
    ('clf',    LogisticRegression(max_iter=500)),
])
correct_fs_scores = cross_val_score(pipe_fs_correct, X_noise, y_noise, cv=cv_fs, scoring='accuracy')

print(f"Leaky (select-then-CV)   — mean accuracy: {leaky_fs_scores.mean():.4f}")
print(f"Correct (select-in-pipe) — mean accuracy: {correct_fs_scores.mean():.4f}")
print(f"True chance level:                        0.5000")
print(f"Optimism from selection leakage:          {leaky_fs_scores.mean() - correct_fs_scores.mean():+.4f}")

**Exercise A2.** The leaky approach above reports accuracy noticeably above
0.50 even though `X_noise` and `y_noise` are entirely independent by
construction. Compare the size of this optimism bias to the scaler-leakage
bias measured in A2, and explain in your own words why feature-selection
leakage is the more dangerous of the two when $p \gg n$.

## Part B — Building and inspecting pipelines

We build pipelines of increasing complexity, inspect their structure, and verify
that the step names give access to fitted parameters.

### B1. A simple two-step pipeline

In [ ]:
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(max_iter=500, C=1.0)),
])

pipe.fit(X_tr, y_tr)
print(f"Test accuracy: {accuracy_score(y_te, pipe.predict(X_te)):.4f}")
print()
print("Fitted scaler mean (first 8 features):")
print(pipe['scaler'].mean_[:8].round(3))
print()
print("Pipeline steps:")
for name, step in pipe.steps:
    print(f"  {name}: {step.__class__.__name__}")

### B2. Adding an intermediate transformer

In [ ]:
from sklearn.decomposition import PCA

pipe_pca = Pipeline([
    ('scaler', StandardScaler()),
    ('pca',    PCA(n_components=20, random_state=2)),
    ('clf',    LogisticRegression(max_iter=500)),
])

pipe_pca.fit(X_tr, y_tr)
acc_pca = accuracy_score(y_te, pipe_pca.predict(X_te))
print(f"Test accuracy (with PCA to 20 components): {acc_pca:.4f}")
print(f"Variance explained by 20 PCs: "
      f"{pipe_pca['pca'].explained_variance_ratio_.sum():.4f}")

In [ ]:
# Compare several pipeline configurations
configs = {
    'Scaler + LR':           Pipeline([('s', StandardScaler()),
                                       ('c', LogisticRegression(max_iter=500))]),
    'Scaler + PCA(20) + LR': pipe_pca,
    'Scaler + RF':           Pipeline([('s', StandardScaler()),
                                       ('c', RandomForestClassifier(n_estimators=100,
                                                                    random_state=3))]),
    'No scaler + RF':        Pipeline([('c', RandomForestClassifier(n_estimators=100,
                                                                    random_state=3))]),
}

cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=4)
print(f"{'Configuration':<30} {'CV mean':>8} {'CV std':>8}")
for name, p in configs.items():
    s = cross_val_score(p, X_tr, y_tr, cv=cv5, scoring='accuracy')
    print(f"{name:<30} {s.mean():>8.4f} {s.std():>8.4f}")

**Exercise B.** Scaling has no effect on a Random Forest (it is invariant to
monotone feature transformations). Confirm this from the table above, and
explain why the result would differ for a logistic regression or an SVM.

## Part C — Column transformers for heterogeneous features

Real datasets combine continuous and categorical features. A `ColumnTransformer`
applies different preprocessing sub-pipelines to different column subsets and
concatenates the results before passing them to the model.

### C1. Constructing a mixed dataset

In [ ]:
import pandas as pd
from sklearn.datasets import make_classification

# Synthetic dataset with mixed types
rng = np.random.default_rng(5)
n = 800

X_cont = rng.standard_normal((n, 4))                         # 4 continuous features
cat_a  = rng.choice(['low', 'medium', 'high'], size=n)       # ordinal (treated as nominal)
cat_b  = rng.choice(['A', 'B', 'C', 'D'],     size=n)       # nominal

# Introduce some missing values in the continuous block
mask = rng.random((n, 4)) < 0.08
X_cont[mask] = np.nan

y_mixed = (X_cont[:, 0] + X_cont[:, 1] > 0).astype(int)
# Fill NaN briefly to compute y, then restore
y_mixed = ((np.where(np.isnan(X_cont[:, 0]), 0, X_cont[:, 0]) +
            np.where(np.isnan(X_cont[:, 1]), 0, X_cont[:, 1])) > 0).astype(int)

df = pd.DataFrame(X_cont, columns=['age', 'income', 'score', 'tenure'])
df['level']    = cat_a
df['category'] = cat_b
df['target']   = y_mixed

print(df.head())
print(f"\nMissing values per column:\n{df.isnull().sum()}")
print(f"\nClass balance: {y_mixed.mean():.3f} positive")

### C2. Building the column transformer pipeline

In [ ]:
numeric_cols     = ['age', 'income', 'score', 'tenure']
categorical_cols = ['level', 'category']

X_mixed = df[numeric_cols + categorical_cols]
y_m     = df['target'].values

X_tr_m, X_te_m, y_tr_m, y_te_m = train_test_split(
    X_mixed, y_m, test_size=0.25, random_state=6, stratify=y_m
)

# Numeric sub-pipeline: impute then scale
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])

# Categorical sub-pipeline: impute then one-hot encode
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ('num', num_pipe, numeric_cols),
    ('cat', cat_pipe, categorical_cols),
])

full_pipe = Pipeline([
    ('prep', preprocessor),
    ('clf',  LogisticRegression(max_iter=500)),
])

full_pipe.fit(X_tr_m, y_tr_m)
acc_mixed = accuracy_score(y_te_m, full_pipe.predict(X_te_m))
print(f"Test accuracy (mixed features, imputation inside pipeline): {acc_mixed:.4f}")

# Check output shape after preprocessing
X_tr_transformed = preprocessor.fit_transform(X_tr_m, y_tr_m)
print(f"\nInput shape:  {X_tr_m.shape}")
print(f"Output shape: {X_tr_transformed.shape}")
print(f"  ({len(numeric_cols)} numeric + {X_tr_transformed.shape[1] - len(numeric_cols)} "
      f"one-hot encoded columns)")

In [ ]:
# Confirm imputer is refitted per fold (not once on full data)
cv5m = StratifiedKFold(n_splits=5, shuffle=True, random_state=7)
scores_mixed = cross_val_score(full_pipe, X_mixed, y_m, cv=cv5m, scoring='accuracy')
print(f"5-fold CV on mixed dataset: {scores_mixed.round(4)}")
print(f"Mean: {scores_mixed.mean():.4f}  Std: {scores_mixed.std():.4f}")

**Exercise C.** In the column transformer above, the imputer for the numeric
columns uses `strategy='median'`. Explain why the median must be computed on the
training fold only in each CV split, and what would go wrong if it were computed
on the full dataset first.

## Part D — Hyperparameter search over a pipeline

The double-underscore naming convention (`stepname__parametername`) lets
`GridSearchCV` tune any parameter in any pipeline step, including preprocessing
choices, simultaneously with model hyperparameters.

In [ ]:
# Search over imputation strategy, PCA components, and classifier C
pipe_search = Pipeline([
    ('scaler', StandardScaler()),
    ('pca',    PCA(random_state=8)),
    ('clf',    LogisticRegression(max_iter=500)),
])

param_grid = {
    'pca__n_components': [10, 20, 30, 40],
    'clf__C':            [0.1, 1.0, 10.0],
}

cv_inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=9)
gs = GridSearchCV(pipe_search, param_grid, cv=cv_inner,
                  scoring='accuracy', n_jobs=-1, refit=True)
gs.fit(X_tr, y_tr)

print(f"Best params:     {gs.best_params_}")
print(f"Best CV score:   {gs.best_score_:.4f}")
print(f"Test accuracy:   {accuracy_score(y_te, gs.predict(X_te)):.4f}")

In [ ]:
# Plot the grid: mean CV accuracy as a function of n_components and C
import pandas as pd

results = pd.DataFrame(gs.cv_results_)
pivot = results.pivot_table(
    index='param_pca__n_components',
    columns='param_clf__C',
    values='mean_test_score'
)

fig, ax = plt.subplots(figsize=(6, 4))
for c_val in pivot.columns:
    ax.plot(pivot.index, pivot[c_val], marker='o', label=f'C={c_val}')
ax.set_xlabel('PCA components')
ax.set_ylabel('Mean CV accuracy')
ax.set_title('Grid search: PCA components × regularisation C')
ax.legend(title='C')
plt.tight_layout()
plt.savefig('/tmp/d1_gridsearch.png', dpi=110)
plt.close()
print("Saved d1_gridsearch.png")

**Exercise D.** The grid search above tunes both a preprocessing step (PCA)
and a model hyperparameter (C) simultaneously. Explain why this gives a better
estimate of the best configuration than first selecting `n_components` by one
CV run and then selecting `C` by a second CV run.

## Part E — Custom transformers and pipeline persistence

Any class implementing `fit` and `transform` can be a pipeline step.
We build two custom transformers, insert them into a pipeline, and then
save and reload the fitted pipeline with `joblib`.

### E1. A percentile clipper

In [ ]:
class PercentileClipper(BaseEstimator, TransformerMixin):
    """Clip each feature to the [lower_pct, upper_pct] percentile range
    computed on the training data, then pass through unchanged values."""

    def __init__(self, lower_pct=5, upper_pct=95):
        self.lower_pct = lower_pct
        self.upper_pct = upper_pct

    def fit(self, X, y=None):
        self.lower_ = np.percentile(X, self.lower_pct, axis=0)
        self.upper_ = np.percentile(X, self.upper_pct, axis=0)
        return self

    def transform(self, X):
        return np.clip(X, self.lower_, self.upper_)


# Verify it can be inserted into a pipeline and cross-validated
pipe_clip = Pipeline([
    ('clip',   PercentileClipper(lower_pct=2, upper_pct=98)),
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(max_iter=500)),
])

scores_clip = cross_val_score(pipe_clip, X_tr, y_tr,
                               cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=10),
                               scoring='accuracy')
print(f"Pipeline with PercentileClipper — CV mean: {scores_clip.mean():.4f}")

### E2. A feature interaction transformer

In [ ]:
class PairwiseProductTransformer(BaseEstimator, TransformerMixin):
    """Append pairwise products of the first `n_pairs` feature pairs to X."""

    def __init__(self, n_pairs=5):
        self.n_pairs = n_pairs

    def fit(self, X, y=None):
        p = X.shape[1]
        # Store the pairs to compute (fixed at fit time so transform is consistent)
        all_pairs = [(i, j) for i in range(p) for j in range(i+1, p)]
        self.pairs_ = all_pairs[:self.n_pairs]
        return self

    def transform(self, X):
        products = np.column_stack([X[:, i] * X[:, j] for i, j in self.pairs_])
        return np.hstack([X, products])


pipe_interact = Pipeline([
    ('scaler',   StandardScaler()),
    ('interact', PairwiseProductTransformer(n_pairs=10)),
    ('clf',      LogisticRegression(max_iter=500)),
])

scores_interact = cross_val_score(
    pipe_interact, X_tr, y_tr,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=11),
    scoring='accuracy'
)
print(f"Pipeline with interaction features — CV mean: {scores_interact.mean():.4f}")

### E3. Saving and loading a fitted pipeline

In [ ]:
# Fit the full grid-search pipeline on the training set
pipe_final = Pipeline([
    ('scaler', StandardScaler()),
    ('pca',    PCA(n_components=gs.best_params_['pca__n_components'], random_state=12)),
    ('clf',    LogisticRegression(C=gs.best_params_['clf__C'], max_iter=500)),
])
pipe_final.fit(X_tr, y_tr)

# Save to disk
save_path = '/tmp/MA3632_week11_pipeline.joblib'
joblib.dump(pipe_final, save_path)
print(f"Pipeline saved to {save_path}")

# Reload and verify predictions are identical
pipe_loaded = joblib.load(save_path)
preds_original = pipe_final.predict(X_te)
preds_loaded   = pipe_loaded.predict(X_te)

print(f"Predictions identical after reload: {np.array_equal(preds_original, preds_loaded)}")
print(f"Test accuracy (loaded pipeline):    {accuracy_score(y_te, preds_loaded):.4f}")

In [ ]:
# Record the sklearn version alongside the saved model — good practice for deployment
import sklearn
version_path = '/tmp/MA3632_week11_pipeline_version.txt'
with open(version_path, 'w') as f:
    f.write(f"scikit-learn=={sklearn.__version__}\n")
    f.write(f"pipeline steps: {[name for name, _ in pipe_final.steps]}\n")

print(open(version_path).read())

**Exercise E.** The `PairwiseProductTransformer` stores `self.pairs_` during
`fit`. Why is it important that `transform` uses `self.pairs_` rather than
recomputing the pairs from the input shape? Construct a concrete example where
recomputing would give incorrect results.

---

## Take-home exercises

**Exercise 1 — Quantifying leakage on a high-dimensional dataset.**
Generate a dataset with `make_classification(n_samples=200, n_features=500,
n_informative=5, random_state=0)`. Compare leaky and correct 5-fold CV accuracy
for a logistic regression. The gap should be much larger than on Digits. Explain
why dimensionality amplifies the leakage bias.

**Exercise 2 — Remainder columns.**
Load the Wine dataset (`load_wine()`). Set up a `ColumnTransformer` that
standardises the first six features and passes the remaining seven through
unchanged (`remainder='passthrough'`). Wrap this in a pipeline with a Random
Forest and evaluate with 5-fold CV. Then compare to a pipeline that standardises
all features. Comment on any difference.

**Exercise 3 — Searching over the imputation strategy.**
Using the mixed dataset from Part C, extend the grid search to include
`prep__num__imputer__strategy` in `['mean', 'median']` alongside at least two
values of the classifier's regularisation parameter. Report the best combination
and verify on the test set.

**Exercise 4 — Custom transformer in a grid search.**
Insert `PercentileClipper` into a pipeline with a logistic regression on the
Digits training set. Add `clip__lower_pct` and `clip__upper_pct` to a
`GridSearchCV` with values $\{1, 5, 10\}$ for each. Report the best clipping
thresholds and whether they improve CV accuracy over no clipping.